# Feature Selection Audit — Including New `context` Column

**Purpose:** Verify which 11 features should be used for HMM analysis by:
1. Loading latest dataset with any new columns (e.g., `context`)
2. Listing ALL available features
3. Auditing the 11 features claimed vs actually used
4. Confirming original paper table vs current notebook
5. Recommending final feature set

**Question:** Should the 11th feature be `tr_overlap_time_s` or `context` or something else?


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports loaded")


✓ Imports loaded


## 1. Load Dataset & List ALL Available Features


In [2]:
# Load the collective features dataset
windows = pd.read_csv('../../_tmp_collective_fullsample_export/features/collective_window_features.tsv', sep='\t')

print(f"✓ Loaded {len(windows)} windows")
print(f"\n📋 FULL COLUMN LIST ({len(windows.columns)} columns):")
print("=" * 70)
for i, col in enumerate(windows.columns, 1):
    dtype = windows[col].dtype
    non_null = windows[col].notna().sum()
    print(f"  {i:2d}. {col:40s} | dtype: {str(dtype):10s} | non-null: {non_null:4d}/{len(windows)}")


✓ Loaded 1521 windows

📋 FULL COLUMN LIST (45 columns):
   1. group_id                                 | dtype: str        | non-null: 1521/1521
   2. task_id                                  | dtype: str        | non-null: 1521/1521
   3. window_index                             | dtype: int64      | non-null: 1521/1521
   4. window_start_s                           | dtype: float64    | non-null: 1521/1521
   5. n_participants                           | dtype: int64      | non-null: 1521/1521
   6. n_physio_valid                           | dtype: int64      | non-null: 1521/1521
   7. group_hr_mean_bpm_mean                   | dtype: float64    | non-null: 1462/1521
   8. group_hr_mean_bpm_std                    | dtype: float64    | non-null: 1366/1521
   9. group_hrv_rmssd_ms_mean                  | dtype: float64    | non-null: 1355/1521
  10. group_hrv_rmssd_ms_std                   | dtype: float64    | non-null: 1151/1521
  11. group_eda_tonic_mean_mean                | dtype

## 2. Compare Three Feature Sets

**Option A: Original Paper Table (11 features)**
**Option B: Current Notebook (11 features)**  
**Option C: With new `context` column (12 features?)**


In [ ]:
print("\n" + "=" * 80)
print("OPTION A: ORIGINAL PAPER TABLE (Your 11-feature list)")
print("=" * 80)
option_a = [
    ('Overlaps (count)', 'tr_overlap_count'),
    ('Laughter (count)', 'tr_laughter_count'),
    ('Active speakers (n)', 'tr_n_active_speakers'),
    ('Backchannel (count)', 'tr_backchannel_count'),
    ('Silence duration (sec)', 'tr_silence_duration_s'),
    ('Competitive OVL (count)', 'tr_competitive_overlap'),
    ('HR (z-scored)', 'group_hr_mean_bpm_mean'),
    ('EDA phasic (z-scored)', 'group_eda_phasic_rate_hz_mean'),
    ('Temp (z-scored)', 'group_temp_mean_mean'),
    ('Pupil diam (z-scored)', 'group_et_pupil_mean_mean'),
    ('PCA dim 1', 'PCA_DIM_1_or_CONTEXT?')
]

for i, (label, col) in enumerate(option_a, 1):
    exists = col in windows.columns if col != 'PCA_DIM_1_or_CONTEXT?' else '?'
    print(f"  {i:2d}. {label:25s} → {col:40s} | exists: {exists}")

print("\n" + "=" * 80)
print("OPTION B: CURRENT NOTEBOOK (11 features)")
print("=" * 80)
option_b = [
    ('HR (z)', 'group_hr_mean_bpm_mean'),
    ('EDA phasic (z)', 'group_eda_phasic_rate_hz_mean'),
    ('Temp (z)', 'group_temp_mean_mean'),
    ('Silence [log]', 'tr_silence_duration_s'),
    ('Backchannels [log]', 'tr_backchannel_count'),
    ('Overlaps [log]', 'tr_overlap_count'),
    ('Competitive OVL [log] ★', 'tr_competitive_overlap'),
    ('Laughter [log]', 'tr_laughter_count'),
    ('Active speakers', 'tr_n_active_speakers'),
    ('Pupil diam (z)', 'group_et_pupil_mean_mean'),
    ('Overlap time [log]', 'tr_overlap_time_s')
]

for i, (label, col) in enumerate(option_b, 1):
    exists = col in windows.columns
    print(f"  {i:2d}. {label:25s} → {col:40s} | exists: {exists}")

print("\n" + "=" * 80)
print("CHECK: Are ALL Option B features present?")
print("=" * 80)
for label, col in option_b:
    if col not in windows.columns:
        print(f"  ❌ MISSING: {col}")
    else:
        non_null = windows[col].notna().sum()
        print(f"  ✓ {col:40s} ({non_null}/{len(windows)} non-null)")

# Check if 'context' column exists
if 'context' in windows.columns:
    print("\n" + "=" * 80)
    print("NEW: `context` COLUMN FOUND")
    print("=" * 80)
    print(f"  Values: {windows['context'].unique()}")
    print(f"  Non-null: {windows['context'].notna().sum()}/{len(windows)}")
else:
    print("\n" + "=" * 80)
    print("⚠️  NO `context` COLUMN found in dataset")
    print("=" * 80)


## 3. Recommendation & Decision


In [3]:
print("\n" + "=" * 80)
print("DECISION MATRIX")
print("=" * 80)

decision_matrix = pd.DataFrame({
    'Set': ['Option A\n(Paper table)', 'Option B\n(Current notebook)', 'Option C\n(+ context)'],
    '# Features': [11, 11, '11 or 12?'],
    '11th feature': ['PCA dim 1 (unclear)', 'tr_overlap_time_s (clear)', 'context (new)'],
    'All present?': ['❓', '✅', '❓'],
    'BIC impact': ['Unknown (old)', 'k=4 selected', 'TBD'],
    'Recommended?': ['⚠️  Verify', '✅ CURRENT', 'ℹ️  If context exists']
})

print(decision_matrix.to_string(index=False))

print("\n" + "=" * 80)
print("NEXT STEPS")
print("=" * 80)
print("""
1. Q: Does `context` column exist in the dataset?
   A: Check cell output above — if not present, use Option B (current)

2. Q: Should 11th feature be `tr_overlap_time_s` or `context`?
   A: 
   - tr_overlap_time_s: Complementary to overlap count (DURATION vs COUNT)
     ✓ Already integrated into current notebook
     ✓ Meaningful multimodal signal
   
   - context: New column, purpose TBD
     ? Need to understand what context represents
     ? May be interesting but requires validation

3. Q: What was the paper table based on?
   A: Clarify if that's:
     - Exploratory analysis (may be outdated)
     - Prior publication (need to align)
     - Different feature set (need rationale for change)

RECOMMENDATION: 
→ If all 11 features in Option B are present and meaningful, KEEP Option B (current)
→ If `context` is important, ADD IT and discuss why (12 features, new BIC selection)
→ PCA dim 1 as 11th feature is NOT recommended (redundant with downstream PCA)
""")



DECISION MATRIX
                         Set # Features              11th feature All present?    BIC impact          Recommended?
     Option A\n(Paper table)         11       PCA dim 1 (unclear)            ❓ Unknown (old)            ⚠️  Verify
Option B\n(Current notebook)         11 tr_overlap_time_s (clear)            ✅  k=4 selected             ✅ CURRENT
       Option C\n(+ context)  11 or 12?             context (new)            ❓           TBD ℹ️  If context exists

NEXT STEPS

1. Q: Does `context` column exist in the dataset?
   A: Check cell output above — if not present, use Option B (current)

2. Q: Should 11th feature be `tr_overlap_time_s` or `context`?
   A: 
   - tr_overlap_time_s: Complementary to overlap count (DURATION vs COUNT)
     ✓ Already integrated into current notebook
     ✓ Meaningful multimodal signal

   - context: New column, purpose TBD
     ? Need to understand what context represents
     ? May be interesting but requires validation

3. Q: What was the 

In [4]:
## 4. Deep Dive: Overlap Categorization Logic

# Load fresh for analysis
windows = pd.read_csv('../../_tmp_collective_fullsample_export/features/collective_window_features.tsv', sep='\t')

# Filter to windows with transcript data
data = windows[windows['tr_spk_duration_s'].notna()].copy()

print("=" * 80)
print("OVERLAP CATEGORIZATION STATISTICS")
print("=" * 80)

overlap_cols = ['tr_overlap_count', 'tr_competitive_overlap', 'tr_cooperative_overlap', 
                'tr_collaborative_overlap', 'tr_needs_review_overlap', 'tr_overlap_time_s']

overlap_stats = data[overlap_cols].describe().T
print("\n", overlap_stats)

print("\n" + "=" * 80)
print("CHECKING: Do overlap subcategories sum to total overlap count?")
print("=" * 80)

# Calculate sum of categorized overlaps
data['sum_categorized'] = (data['tr_competitive_overlap'] + 
                           data['tr_cooperative_overlap'] + 
                           data['tr_collaborative_overlap'] + 
                           data['tr_needs_review_overlap'])

# Compare to raw count
comparison = data[['tr_overlap_count', 'tr_competitive_overlap', 'tr_cooperative_overlap',
                   'tr_collaborative_overlap', 'tr_needs_review_overlap', 'sum_categorized']].copy()

print("\nSample of overlap breakdown (first 10 windows):")
print(comparison.head(10).to_string())

# Check if they sum correctly
match_count = (data['tr_overlap_count'] == data['sum_categorized']).sum()
print(f"\n✓ Categorized overlaps match raw count: {match_count}/{len(data)} windows")

if match_count != len(data):
    print("\n⚠️  NOT a 1-to-1 mapping! Some overlaps are categorized differently or excluded")
    mismatches = data[data['tr_overlap_count'] != data['sum_categorized']]
    print(f"   Mismatches: {len(mismatches)}")
    print(f"   Sample: {mismatches[['tr_overlap_count', 'sum_categorized']].head()}")

print("\n" + "=" * 80)
print("QUESTION: Is `tr_collaborative_overlap` based on TIMING?")
print("=" * 80)

# Check correlation between collaborative overlaps and overlap_time_s
corr = data['tr_collaborative_overlap'].corr(data['tr_overlap_time_s'])
print(f"\nCorrelation(collaborative_overlap, overlap_time_s): {corr:.3f}")

if corr > 0.7:
    print("✅ STRONG correlation! Collaborative overlaps are likely TIMING-BASED")
    print("   (i.e., overlaps lasting longer → classified as collaborative)")
else:
    print("❌ WEAK correlation — collaborative is likely SEMANTIC, not timing-based")

# Also check relationship with total overlap time
corr_total = data['tr_overlap_count'].corr(data['tr_overlap_time_s'])
print(f"Correlation(overlap_count, overlap_time_s): {corr_total:.3f}")

print("\n" + "=" * 80)
print("INSIGHT")
print("=" * 80)
print("""
If collaborative_overlap has HIGH correlation with overlap_time_s:
→ It's timing-based (overlaps lasting >Xms are "collaborative")
→ More objective, less prone to annotator bias
→ GOOD for HMM (reliable signal)

If NO correlation with timing:
→ It's semantically labeled (requires human judgment)
→ May introduce bias, needs validation
→ OKAY for HMM but less reliable than timing-based
""")


OVERLAP CATEGORIZATION STATISTICS

                           count      mean        std  min    25%    50%  \
tr_overlap_count          545.0  1.411009   2.107381  0.0  0.000  1.000   
tr_competitive_overlap    545.0  0.456881   0.946391  0.0  0.000  0.000   
tr_cooperative_overlap    545.0  0.889908   1.647968  0.0  0.000  0.000   
tr_collaborative_overlap  545.0  0.277064   0.837059  0.0  0.000  0.000   
tr_needs_review_overlap   545.0  0.064220   0.317250  0.0  0.000  0.000   
tr_overlap_time_s         545.0  9.166299  19.744212  0.0  0.003  2.535   

                            75%      max  
tr_overlap_count          2.000   15.000  
tr_competitive_overlap    1.000    9.000  
tr_cooperative_overlap    1.000   14.000  
tr_collaborative_overlap  0.000    8.000  
tr_needs_review_overlap   0.000    3.000  
tr_overlap_time_s         9.946  297.105  

CHECKING: Do overlap subcategories sum to total overlap count?

Sample of overlap breakdown (first 10 windows):
    tr_overlap_count  tr